# 23. Differentiable High-Order Markov Models for Spectrum Prediction

Implements the **high-order Markov** framework from:
**"Differentiable High-Order Markov Models for Spectrum Prediction"** (IEEE ICMLCN 2025).

## Paper (adapted to 72h→24h continuous AU%)
- **Composite states:** Last M′ binned values form a state; we use a "smart state" set (distinct states seen in training, capped).
- **Transition matrix P:** Estimated empirically from one-step transitions; then **differentiable** fine-tuning: P is learned (e.g. softmax rows) to minimize prediction loss.
- **Prediction:** Encode input → state distribution s₀; then s₀ P^k gives distribution at step k; expected AU% = s₀ P^k @ value_vector (value per state from training). Output 24 steps.
- **Discretization:** AU% binned into a few levels so we have a finite state space.

## Same setup as 10–22
Data: work_dir/final, 72h→24h. Metrics: MAE, RMSE, MASE. Naive baseline. Same visuals.


In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
from collections import defaultdict
print(f'TensorFlow: {tf.__version__}')


In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.set_visible_devices(gpus, 'GPU')
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f'GPU enabled: {len(gpus)} device(s)')
        with tf.device('/GPU:0'): _ = tf.constant(1)
        print('GPU ready.')
    except RuntimeError as e: print('GPU config:', e)
else: print('No GPU. On Apple Silicon: pip install tensorflow-metal')
USE_GPU = len(gpus) > 0
num_cores = os.cpu_count() or 4
tf.config.threading.set_intra_op_parallelism_threads(num_cores)
tf.config.threading.set_inter_op_parallelism_threads(num_cores)


## Data loading (same as notebook 10)

In [ ]:
work_dir = Path("work_dir")
if not work_dir.exists():
    work_dir = Path("../work_dir")
final_dir = work_dir / "final"
training_dir = final_dir / "training"
testing_dir = final_dir / "testing"
if not final_dir.exists() or not training_dir.exists() or not testing_dir.exists():
    raise FileNotFoundError("work_dir/final/training and testing not found")
class_options = sorted([d.name for d in training_dir.iterdir() if d.is_dir()])
LOOKBACK = 72
FORECAST_HORIZON = 24
print(f"Bands: {class_options}, Lookback={LOOKBACK}, Horizon={FORECAST_HORIZON}")

def load_data_for_band(band_name: str, split: str):
    split_dir = final_dir / split / band_name
    if not split_dir.exists(): return pd.DataFrame()
    dfs = []
    for p in sorted(split_dir.glob("final_*.parquet")):
        try: dfs.append(pd.read_parquet(p))
        except Exception as e: print(f"Error loading {p}: {e}")
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def prepare_next_day_sequences(train_df, test_df, lookback=72, forecast_horizon=24):
    freqs = sorted(train_df["freq_center_ghz"].unique())
    ths = sorted(train_df["threshold_dbm"].unique())
    if len(ths) > 1:
        train_df = train_df[train_df["threshold_dbm"] == ths[0]].copy()
        test_df = test_df[test_df["threshold_dbm"] == ths[0]].copy()
    X_tr, y_tr, X_te, y_te = [], [], [], []
    for freq in freqs:
        tr = train_df[train_df["freq_center_ghz"] == freq].sort_values(["date", "hour"])["au_pct"].values
        te = test_df[test_df["freq_center_ghz"] == freq].sort_values(["date", "hour"])["au_pct"].values
        for i in range(len(tr) - lookback - forecast_horizon + 1):
            X_tr.append(tr[i:i+lookback])
            y_tr.append(tr[i+lookback:i+lookback+forecast_horizon])
        n_test_days = len(te) // forecast_horizon
        for d in range(n_test_days):
            if d == 0:
                inp = tr[-lookback:] if len(tr) >= lookback else np.concatenate([np.zeros(lookback - len(tr)), tr])
            else:
                h = max(0, lookback - d * forecast_horizon)
                inp = np.concatenate([tr[-h:], te[0:d*forecast_horizon]]) if h > 0 else te[d*forecast_horizon - lookback:d*forecast_horizon]
            tgt = te[d*forecast_horizon:(d+1)*forecast_horizon]
            if len(inp) == lookback and len(tgt) == forecast_horizon:
                X_te.append(inp)
                y_te.append(tgt)
    if not X_tr or not X_te:
        return np.array([]), np.array([]), np.array([]), np.array([])
    return np.array(X_tr).reshape(-1, lookback, 1), np.array(y_tr), np.array(X_te).reshape(-1, lookback, 1), np.array(y_te)


In [ ]:
train_data_by_band = {}
test_data_by_band = {}
for band in class_options:
    tr = load_data_for_band(band, "training")
    te = load_data_for_band(band, "testing")
    if not tr.empty and not te.empty:
        train_data_by_band[band] = tr
        test_data_by_band[band] = te
X_train_list, y_train_list, X_test_list, y_test_list = [], [], [], []
for band in class_options:
    if band not in train_data_by_band: continue
    X_tr, y_tr, X_te, y_te = prepare_next_day_sequences(train_data_by_band[band], test_data_by_band[band], LOOKBACK, FORECAST_HORIZON)
    if len(X_tr) > 0 and len(X_te) > 0:
        X_train_list.append(X_tr)
        y_train_list.append(y_tr)
        X_test_list.append(X_te)
        y_test_list.append(y_te)
X_train = np.vstack(X_train_list)
y_train = np.vstack(y_train_list)
X_test = np.vstack(X_test_list)
y_test = np.vstack(y_test_list)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## Discretization and smart-state high-order Markov
Bin AU% into N_BINS levels; composite state = last M_PRIME binned values; build state table from training (cap at MAX_STATES).

In [ ]:
N_BINS = 5
M_PRIME = 24
MAX_STATES = 500

def to_bins(x, bins_edges):
    x = np.clip(x, 0, 100)
    return np.digitize(x, bins_edges, right=False) - 1

bins_edges = np.linspace(0, 100, N_BINS + 1)[1:-1]
X_train_flat = X_train.reshape(-1, LOOKBACK)
y_train_flat = y_train.reshape(-1, FORECAST_HORIZON)
X_train_bin = np.zeros_like(X_train_flat, dtype=np.int32)
for i in range(X_train_flat.shape[0]):
    X_train_bin[i] = np.clip(to_bins(X_train_flat[i], bins_edges), 0, N_BINS - 1)
y_train_bin = np.clip(to_bins(y_train_flat, bins_edges), 0, N_BINS - 1)

state_to_idx = {}
state_list = []
for i in range(X_train_bin.shape[0]):
    tail = tuple(X_train_bin[i, -M_PRIME:].tolist())
    if tail not in state_to_idx:
        if len(state_to_idx) >= MAX_STATES:
            break
        state_to_idx[tail] = len(state_list)
        state_list.append(tail)
L = len(state_list)
print(f'Number of composite states (smart state): {L}')

value_vec = np.zeros(L)
count_val = np.zeros(L)
for i in range(X_train_bin.shape[0]):
    tail = tuple(X_train_bin[i, -M_PRIME:].tolist())
    if tail not in state_to_idx:
        continue
    j = state_to_idx[tail]
    value_vec[j] += y_train_flat[i, 0]
    count_val[j] += 1
value_vec = np.where(count_val > 0, value_vec / np.maximum(count_val, 1), 50.0)

counts = defaultdict(lambda: defaultdict(float))
for i in range(X_train_bin.shape[0]):
    tail = tuple(X_train_bin[i, -M_PRIME:].tolist())
    if tail not in state_to_idx:
        continue
    next_tail = tuple(X_train_bin[i, -M_PRIME+1:].tolist() + [y_train_bin[i, 0]])
    if next_tail in state_to_idx:
        counts[state_to_idx[tail]][state_to_idx[next_tail]] += 1
    else:
        nearest = min(range(L), key=lambda k: sum(a!=b for a,b in zip(state_list[k], next_tail)))
        counts[state_to_idx[tail]][nearest] += 1

P_emp = np.zeros((L, L))
for i in range(L):
    row_sum = sum(counts[i].values())
    if row_sum > 0:
        for j, c in counts[i].items():
            P_emp[i, j] = c / row_sum
    else:
        P_emp[i, i] = 1.0
P_emp = np.float32(P_emp)
value_vec = np.float32(value_vec)
print('Empirical P and value vector built.')

## Differentiable Markov model: trainable P (softmax rows), predict 24 steps

In [ ]:
def encode_state(x_bin, state_to_idx, state_list, M_PRIME):
    tail = tuple(x_bin[-M_PRIME:].tolist())
    if tail in state_to_idx:
        idx = state_to_idx[tail]
        return idx, None
    best_idx = min(range(len(state_list)), key=lambda k: sum(a!=b for a,b in zip(state_list[k], tail)))
    return best_idx, None

def build_markov_dataset(X_bin, state_to_idx, state_list, L):
    s0_list = []
    for i in range(X_bin.shape[0]):
        idx, _ = encode_state(X_bin[i], state_to_idx, state_list, M_PRIME)
        onehot = np.zeros(L, dtype=np.float32)
        onehot[idx] = 1.0
        s0_list.append(onehot)
    return np.array(s0_list)

class DifferentiableMarkovLayer(layers.Layer):
    def __init__(self, P_init, value_vec, **kwargs):
        super().__init__(**kwargs)
        self.P_init = P_init
        self.value_vec = value_vec
        self.L = P_init.shape[0]
    def build(self, input_shape):
        self.logits = self.add_weight('logits', shape=(self.L, self.L),
            initializer=keras.initializers.Constant(np.log(self.P_init + 1e-8)), trainable=True)
        super().build(input_shape)
    def call(self, s0):
        P = tf.nn.softmax(self.logits, axis=-1)
        v = tf.constant(self.value_vec, dtype=tf.float32)
        preds = []
        s = s0
        for _ in range(FORECAST_HORIZON):
            s = tf.matmul(s, P)
            preds.append(tf.squeeze(tf.matmul(s, v[:, None]), axis=-1))
        return tf.stack(preds, axis=1)

s0_train = build_markov_dataset(X_train_bin, state_to_idx, state_list, L)
inp = layers.Input(shape=(L,))
out = DifferentiableMarkovLayer(P_emp, value_vec)(inp)
model = keras.Model(inp, out)
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
model.summary()

In [ ]:
BATCH = 128 if USE_GPU else 32
EPOCHS = 30
history = model.fit(s0_train, y_train, epochs=EPOCHS, batch_size=BATCH, validation_split=0.2, verbose=1)

X_test_flat = X_test.reshape(-1, LOOKBACK)
X_test_bin = np.zeros_like(X_test_flat, dtype=np.int32)
for i in range(X_test_flat.shape[0]):
    X_test_bin[i] = np.clip(to_bins(X_test_flat[i], bins_edges), 0, N_BINS - 1)
s0_test = build_markov_dataset(X_test_bin, state_to_idx, state_list, L)

y_pred_markov = model.predict(s0_test, verbose=0)
y_pred_markov = np.clip(y_pred_markov, 0, 100).astype(np.float32)

def naive_predictor(X, horizon):
    last = X[:, -1, 0]
    return np.tile(last.reshape(-1, 1), (1, horizon))
y_pred_naive = naive_predictor(X_test, FORECAST_HORIZON)

def calculate_mae(y_true, y_pred):
    return mean_absolute_error(y_true.flatten(), y_pred.flatten())
def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true.flatten(), y_pred.flatten()))
def calculate_mase(y_true, y_pred, y_train):
    mae = np.mean(np.abs(y_true.flatten() - y_pred.flatten()))
    scale = np.mean(np.abs(np.diff(y_train.flatten()))) if len(y_train.flatten()) > 1 else 1.0
    return mae / max(scale, 1e-8)

mae_m = calculate_mae(y_test, y_pred_markov)
rmse_m = calculate_rmse(y_test, y_pred_markov)
mase_m = calculate_mase(y_test, y_pred_markov, y_train)
mae_n = calculate_mae(y_test, y_pred_naive)
rmse_n = calculate_rmse(y_test, y_pred_naive)
mase_n = calculate_mase(y_test, y_pred_naive, y_train)

results_df = pd.DataFrame([
    {"Model": "Diff. High-Order Markov", "MAE": mae_m, "RMSE": rmse_m, "MASE": mase_m},
    {"Model": "Naive Baseline", "MAE": mae_n, "RMSE": rmse_n, "MASE": mase_n},
])
results_df["MAE"] = results_df["MAE"].round(4)
results_df["RMSE"] = results_df["RMSE"].round(4)
results_df["MASE"] = results_df["MASE"].round(4)

print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)
print(f"Lookback: {LOOKBACK}h, Forecast: {FORECAST_HORIZON}h")
print(f"Test samples: {len(y_test)}")
print(results_df.to_string(index=False))
display(results_df)

## Analysis and Visualizations (same as notebook 10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = ['MAE', 'RMSE', 'MASE']
colors = ['#2ecc71', '#3498db', '#9b59b6']
for ax, metric, color in zip(axes, metrics, colors):
    vals = results_df[metric].values
    bars = ax.bar(results_df['Model'], vals, color=color, edgecolor='black', linewidth=0.5)
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} by Model')
    ax.tick_params(axis='x', rotation=15)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02 * max(vals), f'{v:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.suptitle('Next-day prediction: Differentiable High-Order Markov', y=1.02, fontsize=12)
plt.show()

naive_mae = results_df[results_df['Model'] == 'Naive Baseline']['MAE'].values[0]
naive_rmse = results_df[results_df['Model'] == 'Naive Baseline']['RMSE'].values[0]
naive_mase = results_df[results_df['Model'] == 'Naive Baseline']['MASE'].values[0]
improvement = results_df[results_df['Model'] != 'Naive Baseline'].copy()
improvement['MAE_imp_%'] = (1 - improvement['MAE'] / naive_mae) * 100
improvement['RMSE_imp_%'] = (1 - improvement['RMSE'] / naive_rmse) * 100
improvement['MASE_imp_%'] = (1 - improvement['MASE'] / naive_mase) * 100
print('Improvement over Naive Baseline (%):')
display(improvement[['Model', 'MAE_imp_%', 'RMSE_imp_%', 'MASE_imp_%']].round(2))
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(improvement))
w = 0.25
ax.bar(x - w, improvement['MAE_imp_%'], w, label='MAE', color='#2ecc71')
ax.bar(x, improvement['RMSE_imp_%'], w, label='RMSE', color='#3498db')
ax.bar(x + w, improvement['MASE_imp_%'], w, label='MASE', color='#9b59b6')
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(improvement['Model'], rotation=15)
ax.set_ylabel('Improvement (%)')
ax.legend()
ax.set_title('Improvement over Naive (positive = better)')
plt.tight_layout()
plt.show()

In [ ]:
hours = np.arange(FORECAST_HORIZON)
n_show = min(3, len(y_test))
fig, axes = plt.subplots(n_show, 1, figsize=(12, 4*n_show))
if n_show == 1: axes = [axes]
for i in range(n_show):
    ax = axes[i]
    ax.plot(hours, y_test[i], 'k--', linewidth=2.5, label='Actual', alpha=0.8)
    ax.plot(hours, y_pred_markov[i], '-', linewidth=1.6, label='Diff. Markov')
    ax.plot(hours, y_pred_naive[i], '-', linewidth=1.2, label='Naive', alpha=0.7)
    ax.set_title(f'Test sample {i+1}: Predicted vs Actual')
    ax.set_xlabel('Hour')
    ax.set_ylabel('AU (%)')
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hours, y_test.mean(axis=0), 'k--', linewidth=2.5, label='Actual (mean)')
ax.plot(hours, y_pred_markov.mean(axis=0), '-', linewidth=1.6, label='Diff. Markov (mean)')
ax.plot(hours, y_pred_naive.mean(axis=0), '-', linewidth=1.2, label='Naive (mean)', alpha=0.7)
ax.set_title('Mean 24h profile: predicted vs actual')
ax.set_xlabel('Hour')
ax.set_ylabel('AU (%)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

mae_per_hour = np.abs(y_test - y_pred_markov).mean(axis=0)
mae_per_hour_n = np.abs(y_test - y_pred_naive).mean(axis=0)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hours, mae_per_hour, '-o', label='Diff. Markov', markersize=4)
ax.plot(hours, mae_per_hour_n, '-o', label='Naive', markersize=4, alpha=0.7)
ax.set_title('MAE by forecast hour')
ax.set_xlabel('Hour')
ax.set_ylabel('MAE (%)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
residuals_m = (y_test - y_pred_markov).flatten()
residuals_naive = (y_test - y_pred_naive).flatten()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(residuals_m, bins=50, color='#3498db', edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_xlabel('Residual')
axes[0].set_ylabel('Count')
axes[0].set_title('Residuals: Diff. Markov')
axes[1].hist(residuals_naive, bins=50, color='#95a5a6', edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Count')
axes[1].set_title('Residuals: Naive')
plt.suptitle('Residual distribution', y=1.02)
plt.tight_layout()
plt.show()
print(f'Residual mean (bias): Diff. Markov = {residuals_m.mean():.4f}, Naive = {residuals_naive.mean():.4f}')
print(f'Residual std:         Diff. Markov = {residuals_m.std():.4f}, Naive = {residuals_naive.std():.4f}')

best_row = results_df[results_df['Model'] == 'Diff. High-Order Markov'].iloc[0]
imp_mae = (1 - best_row['MAE'] / naive_mae) * 100
print(f"\nBest model: Diff. High-Order Markov (MAE={best_row['MAE']:.4f}, RMSE={best_row['RMSE']:.4f}, MASE={best_row['MASE']:.4f}). Improvement over Naive: MAE {imp_mae:+.1f}%.")

### Key insights

- **Paper (ICMLCN 2025):** High-order Markov models use composite states (last M′ observations); transition matrix P is estimated empirically, then **fine-tuned** via gradient-based supervised learning (differentiable P). Smart-state space keeps only states seen in training.
- **Adaptation:** AU% is discretized into bins; composite state = last M′ binned values; value vector = mean next-step AU% per state; prediction = s₀ P^k @ value for k=1..24.
- **Improvement over Naive:** Positive % means the model beats the last-value baseline.
